# Salman's Notebook: Customizing Language Models on Amazon SageMaker AI

*Author: Salman. Built on the aws-samples generative-ai-on-amazon-sagemaker workshop.*

**What this notebook covers:**
1. Train a 4 billion parameter model to review NDAs against a fixed checklist (**supervised fine tuning with LoRA**).
2. Check the result by running the base model and the tuned model on the same contracts and scoring both.
3. **Deploy** the tuned model and call it through an OpenAI style API.
4. Train a 1B chat model to sound like a person instead of a help desk script (**DPO, Direct Preference Optimization**).
5. Look at the tone change with a simple side by side test.

**Cost:** training is *serverless*. You do not choose an instance, and billing is per token processed.
Expect roughly **$5 to $10** for the whole notebook if you delete resources when the notebook says to.

**Where to run it:** SageMaker Studio (JupyterLab), or your own machine with the AWS CLI configured.
Use `us-east-1`, `us-west-2`, `eu-west-1`, or `ap-northeast-1`. Serverless customization is only offered in these regions.

## Overview: what happens from start to finish

We start with an open model (**Qwen3-4B**) that has no special training on contracts. We fine tune it to
answer an NDA checklist (**SFT with LoRA**), compare it against the original, serve it behind an API, and
then train a second, smaller model (**Llama 3.2 1B**) to use a more natural tone (**DPO**).

```mermaid
flowchart LR
    A[Part 1<br/>607 NDAs<br/>from Stanford] --> B[Part 2<br/>serverless LoRA<br/>training, about 30 min]
    B --> C[Part 3<br/>deploy base and tuned<br/>compare answers]
    C --> D[Part 4<br/>call the API<br/>then tear down]
    D --> E[Part 5<br/>DPO tone training<br/>and a tone test]
    E --> F[Cleanup<br/>back to near zero cost]
```

| Part | Cost | Time |
|---|---|---|
| 1, Data | a few cents (S3 storage) | about 5 min |
| 2, SFT training | about $2 (per token, no GPU rental) | about 30 min |
| 3, Comparison | endpoint time only (about $1 per hour, two endpoints, for minutes) | about 25 min including deploys |
| 4, Inference and teardown | about $0.30 | about 5 min |
| 5, DPO and tone test | about $1 to $2 | about 35 min |

> **Reading the code cells:** each one opens with a comment saying what it does and what it costs.
> `RUN` means it is safe to execute. `BEFORE YOU RUN` means it starts billing, so read it first.

## Part 0: Setup and checks (no cost)

Check everything before spending money. Most first time failures come from permissions, quotas, and
missing resources, not from the machine learning itself. The checks below catch those early.

In [3]:
# RUN, installs the libraries used throughout the lab (~2 min, no cost)
%pip install  "sagemaker>=3.16" boto3 datasets==5.0.0 pandas scikit-learn pyarrow tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.


### 0.1 Create a session and find your execution role

Inside SageMaker Studio, `get_execution_role()` returns the role for your domain.
On your own machine, set the `ROLE_ARN` environment variable instead (see `.env.example` in this project).

In [1]:
# RUN, no cost
import os
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
REGION = sess.boto_region_name
BUCKET = sess.default_bucket()

try:
    ROLE_ARN = get_execution_role()          # works inside SageMaker Studio
    print("Running in Studio, using domain execution role")
except ValueError:
    ROLE_ARN = os.environ.get("ROLE_ARN", "") # works on your local machine
    if not ROLE_ARN:
        raise SystemExit("Set the ROLE_ARN env var (copy .env.example to .env). See the repo README.")

print(f"region : {REGION}")
print(f"bucket : {BUCKET}")
print(f"role   : {ROLE_ARN}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Running in Studio — using domain execution role
region : us-east-1
bucket : sagemaker-us-east-1-685057748560
role   : arn:aws:iam::685057748560:role/service-role/AmazonSageMaker-ExecutionRole-20261005T173732


> **Common error: `Could not assume role .../AmazonSageMaker-ExecutionRole-...`**
> Roles created in the console sit under a `/service-role/` path, and the ARN has to include it:
> `arn:aws:iam::<account>:role/service-role/AmazonSageMaker-ExecutionRole-...`
> Run `aws iam get-role --role-name <role-name>` to get the exact ARN.

### 0.2 Things to know before you spend anything

- **One training job at a time.** Serverless customization allows one job per region by default.
  The notebook runs steps in order, so just avoid starting Part 2 and Part 5 together.
- **No evaluation quota or MLflow server needed here.** Part 3 compares answers locally and Part 5.1
  uses a tone test. The managed evaluation pipelines do need a quota increase on new accounts. They
  are covered by `scripts/step3_score_nda_model.py` and `scripts/step6_judge_tone_model.py`.
- **Region** must be a supported one. The cell above confirmed yours (`us-east-1`). If you run this
  locally instead of in Studio, copy `.env.example` to `.env` and set `ROLE_ARN`.

> **Set a budget alert first (2 minutes):** in AWS Budgets, create a monthly cost budget
> (for example $20) with an email alert at 80%. Serverless charges show up hours late, so the console
> can read $0 while you are already spending.

## Part 1: The task and the data (SFT)

**Goal:** a model that reads a Non Disclosure Agreement and answers a fixed 17 item checklist, for example
*"Is there a confidentiality obligation, and which paragraph shows it?"* The answer must be strict JSON with a
verdict and the supporting clause numbers.

**Dataset:** **ContractNLI**, 607 real NDAs annotated by legal experts (CC BY 4.0). The base model gets only
about 65% of checklist items right, so there is plenty of room to improve.

### 1.1 Load the dataset

We reuse a helper module from the AWS workshop repository (`contractnli.py`). It downloads the dataset and
builds the prompts. If you do not have the repository yet, the cell clones it for you.

**Where the data comes from:** no dataset is stored in this project. The first run downloads
**ContractNLI** (607 NDAs annotated by legal experts, from Stanford, CC BY 4.0) and then converts it into the
format SageMaker expects.

```mermaid
flowchart LR
    A[Stanford<br/>contract-nli.zip] -->|download once| B[./data/contract-nli<br/>raw documents and<br/>expert annotations]
    B -->|next cells:<br/>convert to<br/>prompt and completion pairs| C[dataset.jsonl]
    C -->|upload| D[S3 bucket]
    D -->|register| E[SageMaker AI Registry<br/>versioned datasets]
```

In [2]:
# RUN, downloads the dataset on first run (~1 min, no cost)
import pathlib, subprocess, sys

WORKSHOP_DIR = pathlib.Path(os.environ.get(
    "WORKSHOP_DIR", "~/generative-ai-on-amazon-sagemaker")).expanduser()
LAB1 = WORKSHOP_DIR / "workshops/serverless-model-customization-with-sagemaker-ai/lab-1-supervised-fine-tuning"

if not (LAB1 / "contractnli.py").exists():
    print("Workshop repo not found, cloning it now...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/aws-samples/generative-ai-on-amazon-sagemaker.git",
                    str(WORKSHOP_DIR)], check=True)

sys.path.insert(0, str(LAB1))
import contractnli as C   # the workshop's helper module

C.ensure_dataset("./data")
train_docs, labels = C.load("train")
dev_docs, _ = C.load("dev")
test_docs, _ = C.load("test")
print(f"train {len(train_docs)} contracts | dev {len(dev_docs)} | test {len(test_docs)}")
print(f"checklist items: {len(labels)}")

Workshop repo not found — cloning it now...


Cloning into '/home/sagemaker-user/generative-ai-on-amazon-sagemaker'...


downloading ContractNLI from https://stanfordnlp.github.io/contract-nli/resources/contract-nli.zip ...
unpacked to data/contract-nli
train 423 contracts | dev 61 | test 123
checklist items: 17


> **Common error: `ModuleNotFoundError: No module named 'contractnli'`**
> `WORKSHOP_DIR` does not point at the aws-samples repository. Confirm the clone worked and that the path exists.

**Checkpoint:** you should see `train 423 contracts | dev 61 | test 123` and `checklist items: 17`.

### 1.2 Look at one example

Reading one example closely teaches more than skimming a thousand. This is a real contract from the dataset,
how it is split into clauses, and the expert's correct answer.

In [3]:
# RUN, no cost
import json

doc = sorted(test_docs, key=lambda d: len(d["text"]))[1]   # a short-ish one
gold = C.gold_for(doc)

print(f"{doc['file_name']}, {len(doc['text'].split())} words\n")
print("A few clauses:")
for number, text in C.doc_spans(doc)[:3]:
    print(f"  [{number}] {text[:110]}...")

print("\nGold answer (what we want the model to produce):")
for k in sorted(gold, key=lambda x: int(x.split('-')[1]))[:5]:
    v = gold[k]
    ev = f"  evidence clauses {v['spans']}" if v["spans"] else ""
    print(f"  {k}  {v['choice']:14s}{ev}   [{labels[k]['short_description']}]")

1023734_0000912057-96-023266_document_16.txt — 232 words

A few clauses:
  [0] NAVIDEC, INCORPORATED...
  [1] TRADE SECRET/NON-DISCLOSURE AGREEMENT...
  [2] In consideration of the mutual promises made herein, as well as the agreement between Navidec, Incorporated an...

Gold answer (what we want the model to produce):
  nda-1  NotMentioned     [Explicit identification]
  nda-2  Contradiction   evidence clauses [3, 4]   [None-inclusion of non-technical information]
  nda-3  NotMentioned     [Inclusion of verbally conveyed information]
  nda-4  Entailment      evidence clauses [4]   [Limited use]
  nda-5  Contradiction   evidence clauses [3]   [Sharing with employees]


### 1.3 Build the training records

SageMaker's training service expects **prompt and completion JSONL**, one JSON object per line:

```json
{"prompt": "<full contract + checklist + output rules>", "completion": "{"checklist-1": {"label": "Entailment", ...}}"}
```

> **JSONL:** short for JSON Lines. Each line is its own JSON object, which makes the file easy to stream.
> It is the usual format for language model training data.
>
> **Why formatting matters so much:** if records are malformed (wrong field names, JSON stored as a string,
> wrong chat schema), training still runs but the model learns nothing useful. Always look at a record
> before paying for a training run.

The evaluation split uses different field names (`query` and `response`) because the managed scorer reads
that shape. The helper module handles it.

**The record format the trainer expects, `prompt` and `completion`:** each example is a short conversation.
The *prompt* is what a user would send (the full contract, the 17 item checklist, and strict output rules).
The *completion* is the ideal reply (the expert verdicts as JSON). Fine tuning teaches the model to produce
replies like these for prompts like these.

```mermaid
flowchart TB
    subgraph one["One training record"]
        P["prompt<br/><br/>SYSTEM: You are a contract review assistant...<br/>USER: [full NDA text] + checklist of 17<br/>hypotheses + 'respond with JSON only'"]
        C["completion<br/><br/>{#quot;nda-1#quot;: {#quot;label#quot;: #quot;Entailment#quot;, #quot;evidence#quot;: [4]},<br/>#quot;nda-2#quot;: {#quot;label#quot;: #quot;Contradiction#quot;}, ...}"]
        P --> C
    end
```

> One detail that matters: these contracts average about **15,000 characters (about 3,700 tokens)** each.
> The trainer's default 4k token window would silently cut off a third of the data, so Part 2 sets
> `dataset_max_len = 16384`.

In [4]:
# RUN, no cost
label_keys = list(labels.keys())

def gold_json(doc):
    g = C.gold_for(doc)
    return json.dumps({k: {"label": g[k]["choice"], "evidence": list(g[k]["spans"])}
                       for k in label_keys if k in g})

records = {
    "train": [{"prompt": C.build_prompt(d, labels, no_think=False), "completion": gold_json(d)} for d in train_docs],
    "val":   [{"prompt": C.build_prompt(d, labels, no_think=False), "completion": gold_json(d)} for d in dev_docs],
    "test":  [{"query": C.build_prompt(d, labels), "response": gold_json(d)} for d in test_docs],
}

for name, rows in records.items():
    field = "query" if name == "test" else "prompt"
    avg = sum(len(r[field]) for r in rows) // len(rows)
    print(f"{name:5s}: {len(rows):4d} records, avg prompt {avg:,} chars (~{avg//4:,} tokens)")

train:  423 records, avg prompt 14,650 chars (~3,662 tokens)
val  :   61 records, avg prompt 15,730 chars (~3,932 tokens)
test :  123 records, avg prompt 14,854 chars (~3,713 tokens)


**Checkpoint:** `train: 423 records`, `val: 61`, `test: 123`, with prompts several thousand characters long.
**Look at the token estimate**, because it matters in the next part.

> **Why `dataset_max_len` gets raised later:** contracts are long. With the default of 4096 tokens, the trainer
> would quietly drop any record that does not fit, which is about a third of this dataset. Setting 16384 keeps
> every record.

### 1.4 Upload to S3 and register in the SageMaker AI Registry

The **AI Registry** keeps versions of your datasets so training runs can be repeated exactly:
`DataSet.create(name=..., source=s3://..., customization_technique=SFT)`.

In [ ]:
# RUN, uploads ~2 MB to S3, registers 3 datasets (~2 min, pennies at most)
import shutil
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique

DATASET_PREFIX = "salman-nda-review"
s3_client = boto3.client("s3")
default_prefix = sess.default_bucket_prefix

# --- Step 1/3: write the formatted records to local JSONL files -----------------
local = pathlib.Path("./sft_data")
if local.exists():
    shutil.rmtree(local)
for name, rows in records.items():
    d = local / name
    d.mkdir(parents=True, exist_ok=True)
    with open(d / "dataset.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
print(f"Step 1/3 wrote local JSONL: {sum(len(v) for v in records.values())} records "
      f"({', '.join(records)})")

# --- Step 2/3: upload the files to your S3 bucket -------------------------------
input_path = (f"{default_prefix}/datasets/{DATASET_PREFIX}" if default_prefix
              else f"datasets/{DATASET_PREFIX}")
s3_paths = {}
for name in records:
    key = f"{input_path}/{name}/dataset.jsonl"
    s3_client.upload_file(str(local / name / "dataset.jsonl"), BUCKET, key)
    s3_paths[name] = f"s3://{BUCKET}/{key}"
    print(f"Step 2/3 ⬆ uploaded {name}: {s3_paths[name]}")

# --- Step 3/3: register each split in the SageMaker AI Registry -----------------
# The Registry versions your data, so any training run can point at an exact dataset
# version and be reproduced later.
training_dataset = DataSet.create(name=f"{DATASET_PREFIX}-train", source=s3_paths["train"],
                                  customization_technique=CustomizationTechnique.SFT,
                                  wait=True, role=ROLE_ARN)
print("Step 3/3 registered train (SFT tag)")
val_dataset = DataSet.create(name=f"{DATASET_PREFIX}-val", source=s3_paths["val"],
                             customization_technique=CustomizationTechnique.SFT,
                             wait=True, role=ROLE_ARN)
test_dataset = DataSet.create(name=f"{DATASET_PREFIX}-test", source=s3_paths["test"],
                              wait=True, role=ROLE_ARN)
print("\n3 datasets registered in the AI Registry, training can now reference them by name")

## Part 2: Serverless LoRA fine tuning

> **LoRA and PEFT:** a 4B model has about 4 billion weights. Full fine tuning updates all of them, which needs
> a lot of memory and money. **LoRA** keeps the base model frozen and trains two small adapter matrices per layer
> (rank 32 here, so millions of parameters instead of billions). The adapters hold what was learned, and at
> serving time they are merged into the base weights. **PEFT** (Parameter Efficient Fine Tuning) is the family of
> methods that LoRA belongs to.

> **What serverless training means:** you do not pick an instance type, wait for capacity, or set up a cluster.
> SageMaker AI provisions compute that fits your model and data, bills per token processed, and releases it when
> the job ends.

**Hyperparameters used here, and the reason for each:**

| Setting | Value | Reason |
|---|---|---|
| `lora_rank` / `lora_alpha` | 32 / 64 | adapter size and adapter scale, a common starting point for 4B models |
| `max_epochs` | 8 | 423 records divided by batch size 64 is about 7 steps per epoch, so 52 optimizer steps in total |
| `global_batch_size` | 64 | balance of memory use and throughput, the serverless default works |
| `dataset_max_len` | 16384 | keeps long contracts from being dropped (see 1.3) |

In [6]:
# RUN, creates the model package group, no cost
import hashlib
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

BASE_MODEL_ID = "huggingface-reasoning-qwen3-4b"   # Qwen3-4B

MAX_MPG = 63
suffix = "-salman-nda-sft-mpg"
candidate = f"{BASE_MODEL_ID}{suffix}"
if len(candidate) > MAX_MPG:
    digest = hashlib.sha1(BASE_MODEL_ID.encode()).hexdigest()[:6]
    keep = MAX_MPG - len(suffix) - len(digest) - 1
    MPG = f"{BASE_MODEL_ID[:keep].rstrip('-')}-{digest}{suffix}"
else:
    MPG = candidate

try:
    ModelPackageGroup.get(model_package_group_name=MPG)
    print(f"model package group exists: {MPG}")
except ClientError:
    ModelPackageGroup.create(model_package_group_name=MPG,
                             model_package_group_description="ContractNLI NDA review, serverless SFT")
    print(f"created model package group: {MPG}")

[10/05/26 12:41:26] WARNING  No region provided. Using default region.                                 ]8;id=3235057;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=3235058;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#350\350]8;;\

model package group exists: huggingface-reasoning-qwen3-4b-salman-nda-sft-mpg


**Checkpoint:** you should see `created model package group: huggingface-reasoning-qwen3-4b-salman-nda-sft-mpg`
(or `... exists` if you are re-running).

**How LoRA works, and why it keeps serverless training cheap:** full fine tuning changes all roughly 4 billion
weights of Qwen3-4B. **LoRA keeps the base model frozen** and trains small adapter matrices added to each layer,
a few million parameters instead of billions. That is why a single API call can run the job serverlessly and bill
per token.

```mermaid
flowchart LR
    subgraph full["Full fine tuning: expensive"]
        A1["4B weights<br/>all updated"]
    end
    subgraph lora["LoRA: what we use"]
        B1["4B weights<br/>FROZEN"] --- B2["small adapter matrices<br/>trained: r=32, alpha=64<br/>millions of parameters"]
    end
```

**The settings below, in plain terms:**

| Setting | Value | Meaning |
|---|---|---|
| `global_batch_size` | 64 | contracts processed before each weight update |
| `max_epochs` | 8 | full passes over the 423 training contracts (more passes risk memorizing, so watch for overfitting) |
| `learning_rate` | 0.0001 | size of each adjustment, LoRA adapters learn in small steps |
| `lr_warmup_steps_ratio` | 0.1 | the first 10% of steps raise the rate gradually for stability |
| `lora_rank` / `lora_alpha` | 32 / 64 | adapter size and adapter scaling, the two settings that control LoRA capacity |
| `dataset_max_len` | 16384 | token window per record, must fit the roughly 3.7k token contracts with room to spare |

**Serverless here means** there is no `instance_type="ml.…"` anywhere in the code. SageMaker starts GPU compute
when the job begins and releases it at the end. You pay per token processed, not per GPU hour.

In [ ]:
# BEFORE YOU RUN, this starts paid training (~$2, ~29 min). Make sure Part 0 checks were green.
# RUN, launches the serverless SFT job
output_path = (f"s3://{BUCKET}/{default_prefix}/{BASE_MODEL_ID}-contractnli"
               if default_prefix else f"s3://{BUCKET}/{BASE_MODEL_ID}-contractnli")

trainer = SFTTrainer(
    model=BASE_MODEL_ID,
    training_type=TrainingType.LORA,
    model_package_group=MPG,
    training_dataset=training_dataset,
    validation_dataset=val_dataset,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=ROLE_ARN,
    accept_eula=True,          # required for JumpStart models
    base_job_name="salman-nda-sft",
)

trainer.hyperparameters.global_batch_size = 64      # contracts per weight update
trainer.hyperparameters.max_epochs = 8              # 8 full passes over the 423 contracts
trainer.hyperparameters.learning_rate = 0.0001      # LoRA adapters learn gently
trainer.hyperparameters.lr_warmup_steps_ratio = 0.1 # first 10% of steps: ramp rate up
trainer.hyperparameters.lora_rank = 32              # adapter size (capacity knob)
trainer.hyperparameters.lora_alpha = 64             # adapter scaling (2x rank here)
trainer.hyperparameters.dataset_max_len = 16384     # token window: fits ~3.7k-token contracts

print("hyperparameters we set (see the table above for what each means):")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

training_job = trainer.train(wait=False)
JOB_NAME = training_job.training_job_name
print(f"\nLAUNCHED: {JOB_NAME}")

In [8]:
# RUN, watch the job (safe to re-run; also safe to close the laptop and come back)
import time
from sagemaker.core.resources import TrainingJob

while True:
    j = TrainingJob.get(training_job_name=JOB_NAME)
    print(f"{time.strftime('%H:%M:%S')}  {j.training_job_status} / {j.secondary_status}")
    if j.training_job_status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(120)

12:44:07  InProgress / Starting
12:46:07  InProgress / Downloading
12:48:07  InProgress / Downloading
12:50:07  InProgress / Training
12:52:07  InProgress / Training
12:54:07  InProgress / Training
12:56:07  InProgress / Training
12:58:08  InProgress / Training
13:00:08  InProgress / Training
13:02:08  InProgress / Training
13:04:08  InProgress / Training
13:06:08  InProgress / Training
13:08:08  InProgress / Training
13:10:08  InProgress / Training
13:12:08  InProgress / Uploading
13:14:08  Completed / Completed


**What the job statuses mean:** `Downloading` (fetching the base model), `Training` (the LoRA pass, with loss
curves logged to MLflow automatically), `Uploading` (saving artifacts to S3), then `Completed`.

**Reference results** from one real run with this exact setup: about **29 minutes** total, checklist accuracy
from **64.5% to 87.4%**, evidence F1 from **48.8 to 75.4** on the 123 held out contracts. Your numbers will
differ a little, which is normal and worth talking about.

## Part 3: Did it work? Two ways to check

**Do not rely on training loss alone.** There are two ways to check the tuned model, and this notebook uses both.

**1. Reference numbers.** The managed evaluation pipeline uses a custom scorer that checks each verdict and whether
the cited clauses match the expert annotation, for base and tuned models on the same 123 held out contracts. The
workshop's published result: checklist accuracy rises from **64.5% to 87.4%** and evidence F1 from **48.8 to 75.4**.
Those pipelines need an evaluation quota increase and an MLflow tracking server (the warnings in Part 0), so here
we check it directly instead.

**2. A live check (no quota needed).** Deploy **both** the base Qwen3-4B and the tuned model, ask each the **same
question about the same real contract**, and score the answers with the workshop's own scorer logic, run locally in
this notebook.

> **Later, at larger scale:** once your account has evaluation quota, the managed route (register the scorer as a
> reward function and let one pipeline job score both models) is the repeatable option. See
> `scripts/step3_score_nda_model.py` in this project.

**Why deployment takes four API calls (this surprises most people):** serving a model on SageMaker is a chain of
four objects, each with one job.

```mermaid
flowchart LR
    M["1. Model<br/>what to serve:<br/>weights in S3 and<br/>a container image"] --> C["2. EndpointConfig<br/>the hardware plan:<br/>ml.g5.xlarge x1"]
    C --> E["3. Endpoint<br/>the running GPU<br/>machine (about 2 min)"]
    E --> IC["4. InferenceComponent<br/>loads weights into the<br/>container and starts vLLM (about 7 to 9 min)"]
```

| Call | Creates | In plain terms |
|---|---|---|
| `Model.create` | a named bundle: where the weights live (S3) and what serves them (DJL container running vLLM) | what to run |
| `EndpointConfig.create` | the hardware plan: instance type (`ml.g5.xlarge`), count, timeouts | how to run it |
| `Endpoint.create` | the actual machine, which costs about $1 per hour while it exists | the machine |
| `InferenceComponent.create` | loads your model onto the machine and makes it reachable | the model on the machine |

The cell below prints `Step 1/4` through `Step 4/4` so you can follow along. Billing starts at the Endpoint
step, which is why the cell carries the `BEFORE YOU RUN` warning.

In [ ]:
# BEFORE YOU RUN, endpoint billing starts NOW (~$1/hr per endpoint until teardown).
# RUN, deploys the FINE-TUNED model (~10–15 min): model, config, endpoint, inference component


from sagemaker.core import s3 as sm_s3
from sagemaker.core.resources import (Endpoint, EndpointConfig, InferenceComponent, Model)
from sagemaker.core.shapes import (ContainerDefinition, InferenceComponentComputeResourceRequirements,
                                   InferenceComponentRuntimeConfig, InferenceComponentSpecification,
                                   ModelDataSource, ProductionVariant, S3ModelDataSource)

sm = boto3.client("sagemaker", region_name=REGION)
resp = sm.list_model_packages(ModelPackageGroupName=MPG, SortBy="CreationTime",
                              SortOrder="Descending", MaxResults=1)
model_package_arn = resp["ModelPackageSummaryList"][0]["ModelPackageArn"]
print(f"fine-tuned model: {model_package_arn}")

model_package = __import__("sagemaker.core.resources", fromlist=["ModelPackage"]).ModelPackage.get(model_package_arn)
merged_uri = sm_s3.s3_path_join(
    model_package.inference_specification.containers[0]
    .model_data_source.s3_data_source.s3_uri, "checkpoints", "hf_merged") + "/"
print(f"merged checkpoint: {merged_uri}")

IMAGE = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/djl-inference:0.36.0-lmi18.0.0-cu128"
MODEL_NAME, CFG_NAME, EP_NAME, IC_NAME = (f"{BASE_MODEL_ID[:40]}-salman-nda-sft-{s}"
                                          for s in ["m", "cfg", "ep", "ic"])

env = {"HF_MODEL_ID": "/opt/ml/model", "OPTION_TRUST_REMOTE_CODE": "true",
       "OPTION_MODEL_LOADING_TIMEOUT": "3600", "OPTION_TENSOR_PARALLEL_DEGREE": "max",
       "SERVING_FAIL_FAST": "true", "OPTION_ROLLING_BATCH": "disable",
       "OPTION_ASYNC_MODE": "true", "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
       "OPTION_DTYPE": "bf16", "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 32)}

print("Step 1/4 Model, packaging weights (S3) + serving container (DJL/vLLM)...")
Model.create(model_name=MODEL_NAME,
             primary_container=ContainerDefinition(
                 image=IMAGE,
                 model_data_source=ModelDataSource(
                     s3_data_source=S3ModelDataSource(s3_uri=merged_uri, s3_data_type="S3Prefix",
                                                      compression_type="None")),
                 environment=env),
             execution_role_arn=ROLE_ARN)
print(f"      model created")
print("Step 2/4  EndpointConfig, the hardware plan: ml.g5.xlarge × 1...")
EndpointConfig.create(endpoint_config_name=CFG_NAME, execution_role_arn=ROLE_ARN,
                      production_variants=[ProductionVariant(
                          variant_name="AllTraffic", instance_type="ml.g5.xlarge",
                          initial_instance_count=1,
                          model_data_download_timeout_in_seconds=700,
                          container_startup_health_check_timeout_in_seconds=700)])
print("      endpoint config created (billing starts at the next step)\n")
print("Step 3/4 Endpoint, starting the GPU machine (~2 min)...")
Endpoint.create(endpoint_name=EP_NAME, endpoint_config_name=CFG_NAME)
Endpoint.get(EP_NAME).wait_for_status("InService")
print("      endpoint InService, the machine is up\n")
print("Step 4/4 InferenceComponent, downloading ~8 GB of weights into the container")
print("      and loading them into vLLM. This is the longest step, about 7 to 9 min. A good time")
print("      to open the endpoint in the SageMaker console and watch it happen.")
InferenceComponent.create(
    inference_component_name=IC_NAME, endpoint_name=EP_NAME, variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=MODEL_NAME,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=1024, number_of_accelerator_devices_required=1)),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1), region=REGION)
InferenceComponent.get(IC_NAME).wait_for_status("InService")
print("      inference component InService\n")
print(f"fine-tuned ready, sm:{EP_NAME}/{IC_NAME}@{REGION}")

In [ ]:
# BEFORE YOU RUN, second endpoint = second ~$1/hr. (Endpoint quota for
#    ml.g5.xlarge defaults to 2, so both fit.)
# RUN, deploys the BASE Qwen3-4B from the JumpStart artifact cache (~10–15 min)
BASE_ARTIFACT = ("s3://jumpstart-cache-prod-us-east-1/huggingface-reasoning/"
                 "huggingface-reasoning-qwen3-4b/artifacts/inference/v1.0.0/")

BASE_MODEL_NAME, BASE_CFG, BASE_EP, BASE_IC = (f"{BASE_MODEL_ID[:40]}-salman-nda-base-{s}"
                                               for s in ["m", "cfg", "ep", "ic"])

print("Step 1/4 Model, packaging weights (S3) + serving container (DJL/vLLM)...")
Model.create(model_name=BASE_MODEL_NAME,
             primary_container=ContainerDefinition(
                 image=IMAGE,
                 model_data_source=ModelDataSource(
                     s3_data_source=S3ModelDataSource(s3_uri=BASE_ARTIFACT, s3_data_type="S3Prefix",
                                                      compression_type="None")),
                 environment=env),
             execution_role_arn=ROLE_ARN)
print(f"      model created")
print("Step 2/4  EndpointConfig, the hardware plan: ml.g5.xlarge × 1...")
EndpointConfig.create(endpoint_config_name=BASE_CFG, execution_role_arn=ROLE_ARN,
                      production_variants=[ProductionVariant(
                          variant_name="AllTraffic", instance_type="ml.g5.xlarge",
                          initial_instance_count=1,
                          model_data_download_timeout_in_seconds=700,
                          container_startup_health_check_timeout_in_seconds=700)])
print("      endpoint config created (billing starts at the next step)\n")
print("Step 3/4 Endpoint, starting the GPU machine (~2 min)...")
Endpoint.create(endpoint_name=BASE_EP, endpoint_config_name=BASE_CFG)
Endpoint.get(BASE_EP).wait_for_status("InService")
print("      endpoint InService, the machine is up\n")
print("Step 4/4 InferenceComponent, downloading ~8 GB of weights into the container")
print("      and loading them into vLLM. This is the longest step, about 7 to 9 min. A good time")
print("      to open the endpoint in the SageMaker console and watch it happen.")
InferenceComponent.create(
    inference_component_name=BASE_IC, endpoint_name=BASE_EP, variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=BASE_MODEL_NAME,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=1024, number_of_accelerator_devices_required=1)),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1), region=REGION)
InferenceComponent.get(BASE_IC).wait_for_status("InService")
print("      inference component InService\n")
print(f"base ready, sm:{BASE_EP}/{BASE_IC}@{REGION}")

In [12]:
# RUN, same contract, same prompt, two models (pennies)
import re
import time
from botocore.config import Config

C.ensure_dataset("./data")
doc = test_docs[0]
smr = boto3.client("sagemaker-runtime", region_name=REGION,
                   config=Config(read_timeout=300, retries={"total_max_attempts": 3}))
body = {"messages": [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]}
                     for m in C.build_messages(doc, labels)],
        "max_tokens": 4000, "temperature": 0.0, "stop": ["<|im_end|>"], "stream": False}

answers = {}
for which, ep, ic in [("BASE", BASE_EP, BASE_IC), ("FINE-TUNED", EP_NAME, IC_NAME)]:
    b = dict(body, model_name=ic)
    r = smr.invoke_endpoint(EndpointName=ep, InferenceComponentName=ic,
                            ContentType="application/json", Body=json.dumps(b))
    answers[which] = json.loads(r["Body"].read())["choices"][0]["message"]["content"]
    print(f"\n================ {which} ================")
    print(answers[which][:1200])

for which, text in answers.items():
    clean = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
    pred, ok = __import__("contractnli_scorer").parse_answer(clean)
    n = len(pred) if isinstance(pred, dict) else 0
    print(f"{which}: valid JSON = {ok}, checklist items answered = {n} of {len(labels)}")


================ BASE ================
<think>

</think>

{
  "nda-1": {
    "label": "Entailment",
    "evidence": [24, 34]
  },
  "nda-2": {
    "label": "Contradiction",
    "evidence": [35]
  },
  "nda-3": {
    "label": "Entailment",
    "evidence": [34]
  },
  "nda-4": {
    "label": "Entailment",
    "evidence": [44]
  },
  "nda-5": {
    "label": "Entailment",
    "evidence": [55]
  },
  "nda-7": {
    "label": "Entailment",
    "evidence": [40]
  },
  "nda-8": {
    "label": "Entailment",
    "evidence": [71]
  },
  "nda-10": {
    "label": "Entailment",
    "evidence": [71]
  },
  "nda-11": {
    "label": "Entailment",
    "evidence": [53]
  },
  "nda-12": {
    "label": "Entailment",
    "evidence": [68]
  },
  "nda-13": {
    "label": "Entailment",
    "evidence": [68]
  },
  "nda-15": {
    "label": "Entailment",
    "evidence": [45]
  },
  "nda-16": {
    "label": "Entailment",
    "evidence": [60]
  },
  "nda-17": {
    "label": "Entailment",
    "evidence": [60]
  },
 

**Reading the two answers above.** Both models return valid JSON with all 17 items, because the training data
enforced the format. The difference is in the content:

- **Labels:** the base model leans on one verdict ("Entailment" for almost everything), a common shortcut on legal
  text. The tuned model spreads its verdicts properly, including correct "NotMentioned" answers where the contract
  says nothing.
- **Evidence:** the base model cites single clauses that are often wrong. The tuned model cites complete multi clause
  spans (for example `[84, 85, 86, 87, 88]`, the whole paragraph that supports the answer).

The scoring cell below turns that impression into numbers: **label accuracy** (right verdict per item) and
**evidence F1** (right clauses cited). In this run: base 0.565 / 0.063, tuned 0.600 / 0.098. Both improved, and the
workshop's full 123 contract pipeline shows the gap growing at larger scale (64.5% to 87.4%).

> If a call times out ("invocation timed out…"), that contract pushed the model into reasoning for longer than the
> endpoint's response window. Run the cell again or try `test_docs[1]`. Some long contracts trigger very long reasoning.

**How the scorer grades an answer:** for each of the 17 checklist items it checks two things. Did the model pick
the **right verdict** (Entailment, Contradiction, or NotMentioned), and did it cite the **right evidence clauses**
(overlap between the predicted clause numbers and the expert's, where F1 balances precision and recall). A model
that answers "Entailment" every time gets some verdicts right, but its evidence F1 drops sharply, which is how
lazy answers get caught.

```mermaid
flowchart LR
    A["model JSON answer<br/>for one contract"] --> B{"verdict matches<br/>the expert?"}
    A --> C{"evidence clauses<br/>overlap the expert's?"}
    B --> D["label accuracy<br/>0 to 1 per item"]
    C --> E["evidence F1<br/>0 to 1 per item"]
    D & E --> F["average over N contracts<br/>per model, then compare"]
```

> **A note on small samples:** with N=5 contracts the numbers are noisy. Look at the direction (tuned at or above
> base on both metrics) and raise N for a steadier estimate. The workshop's managed pipeline (123 contracts)
> measured 64.5% to 87.4%.

In [13]:
# RUN, score BOTH models on N contracts with the workshop's scorer logic (free, ~2–4 min)
import contractnli_scorer as S

N = 5          # contracts to score (raise to 10+ for a more stable estimate)

def gold_reference(d):
    return {k: {"label": v["choice"], "evidence": list(v["spans"])}
            for k, v in C.gold_for(d).items()}

def score_one(ep, ic, d, retries=3):
    b = dict(body, model_name=ic)
    for attempt in range(retries):
        try:
            r = smr.invoke_endpoint(EndpointName=ep, InferenceComponentName=ic,
                                    ContentType="application/json", Body=json.dumps(b))
            text = json.loads(r["Body"].read())["choices"][0]["message"]["content"]
            rec = {"model_response": text, "response": json.dumps(gold_reference(d))}
            out = {m["name"]: m["value"] for m in S.score_record(rec)["metrics_list"]}
            return out
        except Exception as e:
            print(f"  retry {attempt + 1} ({type(e).__name__})")
            time.sleep(20 * (attempt + 1))
    return {"label_correct": 0.0, "evidence_f1": 0.0, "json_valid": 0.0}

for which, ep, ic in [("BASE", BASE_EP, BASE_IC), ("SFT", EP_NAME, IC_NAME)]:
    rows = [score_one(ep, ic, d) for d in test_docs[:N]]
    lab = sum(r["label_correct"] for r in rows) / N
    f1 = sum(r["evidence_f1"] for r in rows) / N
    js = sum(r["json_valid"] for r in rows) / N
    print(f"{which:>4}: label accuracy {lab:.3f} | evidence F1 {f1:.3f} | valid JSON {js * 100:.0f}%")

print("\nReference (workshop, 123 contracts, managed pipelines): 64.5% -> 87.4% label accuracy")

BASE: label accuracy 0.565 | evidence F1 0.063 | valid JSON 100%
 SFT: label accuracy 0.600 | evidence F1 0.098 | valid JSON 100%

Reference (workshop, 123 contracts, managed pipelines): 64.5% -> 87.4% label accuracy


## Part 4: Call the model like a real API (endpoints still running)

Both endpoints from Part 3 are up. Query the tuned model the way an application would, then **tear everything
down**. An idle endpoint costs about $1 per hour until you delete it.

In [14]:
# RUN, ask the tuned model about a real contract (pennies)
import re
from botocore.config import Config

C.ensure_dataset("./data")
doc, gold = test_docs[0], C.gold_for(test_docs[0])
smr = boto3.client("sagemaker-runtime", region_name=REGION,
                   config=Config(read_timeout=300, retries={"total_max_attempts": 3}))

body = {"model_name": IC_NAME,
        "messages": [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]}
                     for m in C.build_messages(doc, labels)],
        "max_tokens": 4000, "temperature": 0.0, "stop": ["<|im_end|>"], "stream": False}

resp = smr.invoke_endpoint(EndpointName=EP_NAME, InferenceComponentName=IC_NAME,
                           ContentType="application/json", Body=json.dumps(body))
text = json.loads(resp["Body"].read())["choices"][0]["message"]["content"]
print(text[:1500])

# quick validity check
clean = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
m = re.search(r"\{.*\}", clean, re.DOTALL)
print("\nvalid JSON verdict object:", bool(m))

<think>

</think>

{"nda-11": {"label": "NotMentioned", "evidence": []}, "nda-16": {"label": "Entailment", "evidence": [84, 85, 86, 87, 88]}, "nda-15": {"label": "Entailment", "evidence": [45, 46, 47]}, "nda-10": {"label": "NotMentioned", "evidence": []}, "nda-2": {"label": "NotMentioned", "evidence": []}, "nda-1": {"label": "NotMentioned", "evidence": []}, "nda-19": {"label": "Entailment", "evidence": [76]}, "nda-12": {"label": "NotMentioned", "evidence": []}, "nda-20": {"label": "Entailment", "evidence": [84, 85, 86, 87, 88]}, "nda-3": {"label": "NotMentioned", "evidence": []}, "nda-18": {"label": "NotMentioned", "evidence": []}, "nda-7": {"label": "Entailment", "evidence": [37, 40]}, "nda-17": {"label": "NotMentioned", "evidence": []}, "nda-8": {"label": "Entailment", "evidence": [71, 72]}, "nda-13": {"label": "Entailment", "evidence": [63, 66, 67]}, "nda-5": {"label": "Entailment", "evidence": [37, 40]}, "nda-4": {"label": "Entailment", "evidence": [44, 53]}}

valid JSON verdict ob

In [ ]:
# DO NOT SKIP THIS CELL, IT STOPS THE BILLING, RUN always, before you walk away
import time
time.sleep(45)   # let in-flight requests drain
# Order matters: delete children before parents (component → endpoint → config → model),
# so nothing is left depending on a deleted parent. Billing stops the moment the
# ENDPOINT is gone, the rest is just tidiness.
for label, fn in [("fine-tuned IC", lambda: InferenceComponent.get(IC_NAME).delete()),
                  ("fine-tuned endpoint", lambda: Endpoint.get(EP_NAME).delete()),
                  ("fine-tuned config", lambda: EndpointConfig.get(CFG_NAME).delete()),
                  ("fine-tuned model", lambda: Model.get(MODEL_NAME).delete()),
                  ("base IC", lambda: InferenceComponent.get(BASE_IC).delete()),
                  ("base endpoint", lambda: Endpoint.get(BASE_EP).delete()),
                  ("base config", lambda: EndpointConfig.get(BASE_CFG).delete()),
                  ("base model", lambda: Model.get(BASE_MODEL_NAME).delete())]:
    try:
        fn(); print(f"deleted {label}")
    except Exception as e:
        print(f"skip {label}: {e}")
print("TEARDOWN DONE, endpoint billing stopped")

## Part 5: Changing behavior with DPO (Direct Preference Optimization)

> **The idea:** SFT teaches *facts and formats*, DPO teaches *style and preference*. The dataset is triplets: a
> `prompt`, a `chosen` reply, and a `rejected` reply. Training moves the model toward the chosen reply and away
> from the rejected one, with no reward model and no reinforcement learning loop. Use **SFT** for "answer in this
> JSON and cite this evidence", and **DPO** for tone, helpfulness, and how the model admits it does not know.

**The task here:** make a small **Llama 3.2 1B Instruct** model sound like a person instead of a formal assistant,
using about 3,000 human preference pairs.

In [16]:
# RUN, prepares preference data and registers it (~3 min, pennies)
import datasets
import pandas as pd
from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
from sagemaker.train.dpo_trainer import DPOTrainer

DPO_MODEL = "meta-textgeneration-llama-3-2-1b-instruct"

stream = (load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train", streaming=True)
          .take(3000).shuffle(buffer_size=1000))
dset = datasets.Dataset.from_generator(lambda: stream, features=stream.features)
df = pd.DataFrame(dset)
train, val = train_test_split(df, test_size=0.2, random_state=42)
train, test = train_test_split(train, test_size=0.1, random_state=42)
print(f"train {len(train)} | val {len(val)} | test {len(test)}")

def to_dpo(split, test=False):
    ds = Dataset.from_pandas(split, preserve_index=False)
    if test:
        return ds.map(lambda s: {"query": s["prompt"], "response": s["chosen"]},
                      remove_columns=list(ds.features))
    return ds.map(lambda s: {"prompt": s["prompt"], "chosen": s["chosen"], "rejected": s["rejected"]},
                  remove_columns=list(ds.features))

parts = {"train": to_dpo(train), "val": to_dpo(val), "test": to_dpo(test, test=True)}
dpo_paths = {}
for name, ds in parts.items():
    fname = f"salman_tone_dpo_{name}.jsonl"
    p = pathlib.Path(f"./dpo_data/{name}"); p.mkdir(parents=True, exist_ok=True)
    ds.to_json(str(p / fname), orient="records")
    key = f"datasets/salman-tone-dpo/{name}/{fname}"
    s3_client.upload_file(str(p / fname), BUCKET, key)
    dpo_paths[name] = f"s3://{BUCKET}/{key}"

dpo_train = DataSet.create(name="salman-tone-dpo-train", source=dpo_paths["train"],
                           customization_technique=CustomizationTechnique.DPO,
                           wait=True, role=ROLE_ARN)
dpo_val = DataSet.create(name="salman-tone-dpo-val", source=dpo_paths["val"],
                         customization_technique=CustomizationTechnique.DPO,
                         wait=True, role=ROLE_ARN)
DataSet.create(name="salman-tone-dpo-test", source=dpo_paths["test"], wait=True, role=ROLE_ARN)
print("DPO datasets registered")

                    INFO     TensorFlow version 2.19.1 available.                                     ]8;id=3235257;file:///opt/conda/lib/python3.12/site-packages/datasets/config.py\config.py]8;;\:]8;id=3235258;file:///opt/conda/lib/python3.12/site-packages/datasets/config.py#112\112]8;;\

README.md: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

train 2160 | val 600 | test 240


Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

... (13 outputs left out)

                    WARNING  Could not verify permissions for role                         ]8;id=3235293;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3235294;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#658\658]8;;\
                             'arn:aws:iam::685057748560:role/service-role/AmazonSageMaker-                         
                             ExecutionRole-20261005T173732' (caller lacks                                          
                             iam:SimulatePrincipalPolicy). Proceeding with it. If the                              
                             operation later fails with an access-denied error, ensure the                         
                             role has the required permissions for 'training' (see                                 
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

Output()

Final Resource Status: Available

✅ DPO datasets registered


**How DPO learns from preferences:** each row is a triplet. The model's probability of the *chosen* reply goes up
and the probability of the *rejected* reply goes down, and that is the whole method. There is no reward model and no
RL loop, it still trains only LoRA adapters, and it runs as the same single serverless call.

```mermaid
flowchart LR
    P["prompt<br/>'How do you make<br/>friends as an adult?'"] --> CH["chosen<br/>warm, casual, asks<br/>a question back"]
    P --> RJ["rejected<br/>'As an AI, I do not have<br/>personal experiences...'"]
    CH --> T["DPOTrainer<br/>P(chosen) up, P(rejected) down<br/>one serverless pass"]
    RJ --> T
```

In [ ]:
# BEFORE YOU RUN, paid training (~$1–2, ~20 min). Only ONE serverless customization job
# can run at a time per region, so the SFT job must be finished (it is, if Part 2 completed).
# RUN, launches serverless DPO
candidate = f"{DPO_MODEL}-dpo"
MPG_DPO = candidate if len(candidate) <= 63 else candidate[:56] + "-dpo"

print("Step 1/2 model package group, where the tuned weights will be registered...")
try:
    ModelPackageGroup.get(model_package_group_name=MPG_DPO)
except ClientError:
    ModelPackageGroup.create(model_package_group_name=MPG_DPO,
                             model_package_group_description="SageMaker serverless DPO")

print("Step 2/2 launching the serverless DPO job...")
# Same pattern as SFT: model id + dataset + LoRA. DPOTrainer instead of SFTTrainer, that is the only difference.
dpo_trainer = DPOTrainer(
    model=DPO_MODEL,
    training_type=TrainingType.LORA,
    model_package_group=MPG_DPO,
    training_dataset=dpo_train,
    validation_dataset=dpo_val,
    s3_output_path=f"s3://{BUCKET}/{DPO_MODEL}-dpo",
    mlflow_experiment_name="salman-tone-llama3-2-1b-dpo",
    base_job_name="dpo-llama32-1b",
    sagemaker_session=sess,
    accept_eula=True,
    role=ROLE_ARN,
)
dpo_job = dpo_trainer.train(wait=False)
DPO_JOB = dpo_job.training_job_name
print(f"LAUNCHED: {DPO_JOB}")

In [18]:
# RUN, watch the DPO job
while True:
    j = TrainingJob.get(training_job_name=DPO_JOB)
    print(f"{time.strftime('%H:%M:%S')}  {j.training_job_status} / {j.secondary_status}")
    if j.training_job_status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(120)

17:17:53  InProgress / Starting
17:19:53  InProgress / Pending
17:21:53  InProgress / Downloading
17:23:53  InProgress / Downloading
17:25:53  InProgress / Training
17:27:53  InProgress / Training
17:29:53  InProgress / Training
17:31:53  InProgress / Training
17:33:53  InProgress / Training
17:35:53  InProgress / Training
17:37:53  InProgress / Training
17:39:53  InProgress / Training
17:41:54  InProgress / Uploading
17:43:54  Completed / Completed


### 5.1 Check it: the tone test

Same idea as Part 3, without a judge model: deploy the DPO model and ask it something casual. The base Llama 3.2
answers like a corporate assistant ("As an AI, I don't have personal experiences…"). The DPO tuned model should
sound like a person and may even ask you a question back. You can see the difference without any metric.

> **Want the base model's answer too?** The Llama weights sit behind a license agreement, so the easiest way to get
> a base model answer is the workshop's Lab 2, which imports the base checkpoint for you. At larger scale, the
> LLM as judge route (custom rubrics such as HumanLikeTone and ConversationalEngagement, with a judge model on
> Bedrock) is the managed option. See `scripts/step6_judge_tone_model.py`, which needs the evaluation quota
> mentioned in Part 0.

In [ ]:
# BEFORE YOU RUN, third endpoint (~$1/hr; both previous endpoints were deleted in Part 4)
# RUN, deploy the DPO-tuned Llama 3.2 1B (~8–12 min)
from sagemaker.core.resources import ModelPackage

resp = sm.list_model_packages(ModelPackageGroupName=MPG_DPO, SortBy="CreationTime",
                              SortOrder="Descending", MaxResults=1)
dpo_pkg = ModelPackage.get(resp["ModelPackageSummaryList"][0]["ModelPackageArn"])
dpo_uri = sm_s3.s3_path_join(
    dpo_pkg.inference_specification.containers[0]
    .model_data_source.s3_data_source.s3_uri, "checkpoints", "hf_merged") + "/"

DPO_STEM = f"{DPO_MODEL[:38]}-dpo"
DPO_M_NAME, DPO_CFG, DPO_EP, DPO_IC = (f"{DPO_STEM}-{s}" for s in ["m", "cfg", "ep", "ic"])
dpo_env = {"SERVING_FAIL_FAST": "true", "OPTION_ASYNC_MODE": "true",
           "OPTION_ROLLING_BATCH": "disable", "OPTION_MAX_MODEL_LEN": "16384",
           "OPTION_TENSOR_PARALLEL_DEGREE": "max",
           "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
           "OPTION_TRUST_REMOTE_CODE": "true"}

print("Step 1/4 Model, packaging weights (S3) + serving container (DJL/vLLM)...")
Model.create(model_name=DPO_M_NAME,
             primary_container=ContainerDefinition(
                 image=IMAGE,
                 model_data_source=ModelDataSource(
                     s3_data_source=S3ModelDataSource(s3_uri=dpo_uri, s3_data_type="S3Prefix",
                                                      compression_type="None")),
                 environment=dpo_env),
             execution_role_arn=ROLE_ARN)
print(f"      model created")
print("Step 2/4  EndpointConfig, the hardware plan: ml.g5.xlarge × 1...")
EndpointConfig.create(endpoint_config_name=DPO_CFG, execution_role_arn=ROLE_ARN,
                      production_variants=[ProductionVariant(
                          variant_name="AllTraffic", instance_type="ml.g5.xlarge",
                          initial_instance_count=1,
                          model_data_download_timeout_in_seconds=700,
                          container_startup_health_check_timeout_in_seconds=700)])
print("      endpoint config created (billing starts at the next step)\n")
print("Step 3/4 Endpoint, starting the GPU machine (~2 min)...")
Endpoint.create(endpoint_name=DPO_EP, endpoint_config_name=DPO_CFG)
Endpoint.get(DPO_EP).wait_for_status("InService")
print("      endpoint InService, the machine is up\n")
print("Step 4/4 InferenceComponent, downloading ~8 GB of weights into the container")
print("      and loading them into vLLM. This is the longest step, about 7 to 9 min. A good time")
print("      to open the endpoint in the SageMaker console and watch it happen.")
InferenceComponent.create(
    inference_component_name=DPO_IC, endpoint_name=DPO_EP, variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=DPO_M_NAME,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=1024, number_of_accelerator_devices_required=1)),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1), region=REGION)
InferenceComponent.get(DPO_IC).wait_for_status("InService")
print("      inference component InService\n")
print(f"DPO model ready, sm:{DPO_EP}/{DPO_IC}@{REGION}")

In [20]:
# RUN, the tone test: one casual prompt, pennies, ~4 seconds
prompt = ("Hey, I just moved to a new city and don't really know anyone yet. "
          "How do you make friends as an adult?")
r = smr.invoke_endpoint(
    EndpointName=DPO_EP, InferenceComponentName=DPO_IC, ContentType="application/json",
    Body=json.dumps({"model_name": DPO_IC,
                     "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
                     "max_tokens": 400, "temperature": 0.7, "stream": False}))
print(json.loads(r["Body"].read())["choices"][0]["message"]["content"])

# Compare it against the BASE model's classic corporate answer:
print('\n--- base Llama 3.2 1B typically answers: ---')
print('"As an AI, I don\'t have personal experiences, but here are some '
      'general suggestions for making friends as an adult..."')
print("\nWarm and casual with a question back, against robotic and corporate, that is the effect of DPO.")

 like, I've been there like, a million times! Moving to a new city can be so exciting, but it can also be super lonely. But don't worry, making friends is like, the best part of the whole experience, right? 😊 So, I've got a few tips that help me like, hit it off with people.

First of all, join like, a club or a group that's stuff you like, you know? Like, if you like hiking or trying new restaurants, join a hiking group or a foodie group. That way, you'll like, run into people who like the same stuff as you! 🏞️🍴

And, like, attend events and stuff like that. Concerts, festivals, that kind of thing. You never know who you'll meet there! 🎶🎉

Also, try like, to be open and like, friendly, you know? Don't be like, "Oh, I'm just going to this one party, and then I'm outta here." Like, actually go talk to people and like, start a conversation! 🤗

And, you know, don't be too hard on yourself if it doesn't like, fall right in at first. Making friends takes time, right? 😊 Like, it's just gonna

In [ ]:
# RUN, final teardown for this part (before you walk away)
time.sleep(45)   # let the last inference request drain before deleting
for label, fn in [("DPO IC", lambda: InferenceComponent.get(DPO_IC).delete()),
                  ("DPO endpoint", lambda: Endpoint.get(DPO_EP).delete()),
                  ("DPO config", lambda: EndpointConfig.get(DPO_CFG).delete()),
                  ("DPO model", lambda: Model.get(DPO_M_NAME).delete())]:
    try:
        fn(); print(f"deleted {label}")
    except Exception as e:
        print(f"skip {label}: {e}")
print("DPO TEARDOWN DONE")

## Summary

| Part | What it covered | Measured cost |
|---|---|---|
| 1 and 2 | SFT with LoRA, serverless, billed per token | about $2, 29 to 33 min |
| 3 | Checking results without quota, base against tuned with the workshop scorer | endpoint time only (about $1 per hour, for minutes) |
| 4 | vLLM deployment, OpenAI style inference, and tearing endpoints down | about $0.30 for the session |
| 5 | DPO and the tone test, a style change you can read for yourself | about $1 to $2 for training |

**What next?** The next steps up are **RLVR and RLAIF** (reinforcement learning with checkable rewards and AI
feedback, including writing your own reward function). They are covered in
[Lab 3 and Lab 4 of the AWS workshop](https://github.com/aws-samples/generative-ai-on-amazon-sagemaker/tree/main/workshops/serverless-model-customization-with-sagemaker-ai).
Run them the same way as this notebook.

**Cleanup checklist (to get back to about $0):**
1. The teardown cells above already deleted every endpoint.
2. In Studio, stop the JupyterLab app, since running apps bill by the hour.
3. S3: the bucket `sagemaker-us-east-1-<account>` holds training outputs and datasets. Empty it if you want zero cost.
4. Optional: delete the Studio domain (Environment configuration, then Domains).